# CLIP — Contrastive Language-Image Pretraining

**Paper:** Radford et al. (OpenAI, 2021) — *Learning Transferable Visual Models From Natural Language Supervision*

**Topics:**
1. Motivation — why a joint vision-language space?
2. CLIP Architecture — dual encoder
3. Contrastive loss (InfoNCE) — full derivation
4. Training on 400M image-text pairs (WIT dataset)
5. Zero-shot image classification
6. ALIGN and OpenCLIP variants
7. Code demo + visualisation
8. Interview Q&A

## 1. Motivation — The Problem CLIP Solves

### Traditional Computer Vision
Supervised image classification trains on fixed label sets:
- ImageNet: 1,000 classes (cat, dog, tench, ...) trained with cross-entropy
- The model learns to map images to one of 1,000 fixed categories
- If the category is not in the 1,000 labels, the model fails

**Problems:**
1. **Label bottleneck:** Creating labelled datasets is expensive
2. **Fixed vocabulary:** Cannot classify an image as 'a dog playing in snow' — only 'dog'
3. **No transferability:** An ImageNet classifier must be fully re-trained for medical imaging

### CLIP's Insight
The internet is full of images with natural language descriptions — captions, alt-text, tweets.
What if we train a model to learn **which text matches which image** from 400 million
naturally occurring image-caption pairs?

Key observation:
> If a model can reliably match 'a photo of a dog' to the correct image out of millions,
> it must understand both visual content and language deeply.

The result is a **joint embedding space** where images and text about the same concept
are placed near each other.

## 2. CLIP Architecture — Dual Encoder

CLIP consists of exactly two components:

```
Image I  →  [Vision Encoder]  → image embedding  v ∈ R^d
Text  T  →  [Text Encoder]    → text  embedding  u ∈ R^d
```

**Vision Encoder:** ResNet or Vision Transformer (ViT)
  - Processes a raw image into a fixed-length vector $v \in \mathbb{R}^d$

**Text Encoder:** Transformer (similar to GPT)
  - Processes a text caption into a fixed-length vector $u \in \mathbb{R}^d$

Both encoders project into the **same $d$-dimensional space** (e.g., $d=512$ or $d=1024$).

The key criterion: for a matching (image, text) pair, $\cos(v, u)$ should be high.
For a non-matching pair, $\cos(v, u)$ should be low.

This is achieved by the **InfoNCE contrastive loss** (next section).

## 3. InfoNCE Contrastive Loss — Full Derivation

### Training Setup
At each step, process a batch of $N$ (image, text) pairs:
```
(I_1, T_1), (I_2, T_2), ..., (I_N, T_N)
```
Encode all images: $v_1, v_2, \ldots, v_N \in \mathbb{R}^d$

Encode all texts:  $u_1, u_2, \ldots, u_N \in \mathbb{R}^d$

### Similarity Matrix
Compute pairwise cosine similarities — one entry for every (image $i$, text $j$) pair:

$$S_{ij} = \frac{v_i \cdot u_j}{\|v_i\| \|u_j\|} \times \tau^{-1}$$

Where $\tau$ (tau) is a **learnable temperature parameter** that scales the logits.
- Small $\tau$: sharper distribution — model very confident about matches
- Large $\tau$: flatter distribution — model less decisive

This gives us an $N \times N$ matrix. The **correct pairs are on the diagonal** ($S_{ii}$).

### Loss Computation
We want:
- Each image embedding $v_i$ to be most similar to ITS matched text $u_i$ (row-wise softmax)
- Each text embedding $u_j$ to be most similar to ITS matched image $v_j$ (column-wise softmax)

**Image→Text loss (row direction):**

$$\mathcal{L}_{I \to T} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(S_{ii})}{\sum_{j=1}^{N} \exp(S_{ij})}$$

**Text→Image loss (column direction):**

$$\mathcal{L}_{T \to I} = -\frac{1}{N} \sum_{j=1}^{N} \log \frac{\exp(S_{jj})}{\sum_{i=1}^{N} \exp(S_{ij})}$$

**Total CLIP loss:**

$$\mathcal{L}_{CLIP} = \frac{1}{2}(\mathcal{L}_{I \to T} + \mathcal{L}_{T \to I})$$

**Intuition:**
- For image $I_3$: its correct text $T_3$ should have the highest similarity score out of $N$ texts
- For text $T_3$: its correct image $I_3$ should have the highest similarity score out of $N$ images
- Each training step creates $N^2 - N$ implicit negative pairs (all off-diagonal)
- With $N=32768$ (CLIP's batch size), each step creates ~1 billion negatives!

### Temperature $\tau$ — The Critical Hyperparameter

$$S_{ij} = \frac{v_i \cdot u_j}{\|v_i\| \cdot \|u_j\|} \cdot e^{-\ln \tau}$$

CLIP initialises $\log \tau = 0.07$ and learns it during training.
- If $\tau$ too high: soft probabilities → model learns slowly
- If $\tau$ too low: gradient vanishes for non-maximum positives
- Learned $\tau$ finds the right balance automatically

## 4. Zero-Shot Image Classification

After training, CLIP can classify images into ANY set of categories — without retraining.

**Procedure:**
1. For each class label, create a text prompt: 'a photo of a {class}'
2. Encode all prompts → text embeddings $u_1, \ldots, u_K$
3. Encode the query image → image embedding $v$
4. Predict: $\hat{y} = \arg\max_k \cos(v, u_k)$

**Why 'a photo of a {class}'?** CLIP was trained on web captions, not just class names.
Wrapping labels in natural language prompts bridges the gap between training distribution
and zero-shot inference. This is called **prompt engineering**.

**Results:** CLIP achieves 75.5% top-1 accuracy on ImageNet zero-shot — matching
ResNet-50 trained with full supervision on ImageNet!

## 5. ALIGN and OpenCLIP Variants

### ALIGN (Google, 2021)
- Scale over quality: trains on 1.8 billion (noisy) image-text pairs vs CLIP's 400M (filtered)
- Shows that scale compensates for label noise — noisy but HUGE data > clean but small data
- Uses EfficientNet (image) + BERT (text) as encoders
- Achieves slightly better zero-shot performance than CLIP despite noisier data

### OpenCLIP
- Open-source reimplementation of CLIP (LAION, 2022)
- Trained on LAION-2B (2 billion publicly crawled image-text pairs)
- Models available: ViT-H/14, ViT-G/14 — outperform original OpenAI CLIP
- Key contribution: reproducibility and open weights for research

| Model | Training data | Zero-shot ImageNet |
|---|---|---|
| CLIP ViT-L/14 | WIT-400M (filtered) | 75.5% |
| ALIGN EfficientNet-L2 | 1.8B (noisy) | 76.4% |
| OpenCLIP ViT-H/14 | LAION-2B | 78.0% |
| OpenCLIP ViT-G/14 | LAION-2B | 80.1% |

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(42)

# ── Simulate CLIP contrastive training on a tiny batch ──
N = 5   # batch size
D = 8   # embedding dim

# Ground-truth matched pairs (on diagonal of similarity matrix)
image_captions = [
    ('cat on mat',      'A fluffy cat sitting on a colourful mat'),
    ('dog in park',     'A golden retriever running in a sunny park'),
    ('eiffel tower',    'The Eiffel Tower at night with lights'),
    ('pizza',           'A delicious margherita pizza on a wooden board'),
    ('mountain sunset', 'A dramatic sunset over a mountain range'),
]

# Simulate embeddings such that matched pairs are closer
def make_embeddings(N, D, noise=0.3):
    # Base concept vectors
    base = np.random.randn(N, D)
    base /= np.linalg.norm(base, axis=1, keepdims=True)
    # Image embeddings = base + noise
    v = base + np.random.randn(N, D) * noise
    v /= np.linalg.norm(v, axis=1, keepdims=True)
    # Text embeddings = base + noise (different noise, same base)
    u = base + np.random.randn(N, D) * noise
    u /= np.linalg.norm(u, axis=1, keepdims=True)
    return v, u

v, u = make_embeddings(N, D)
tau = 0.07   # temperature

# Compute similarity matrix  S[i,j] = cos(v_i, u_j) / tau
S = (v @ u.T) / tau
print('Similarity matrix S (N x N):')
labels = [x[0] for x in image_captions]
print(f'{"":12}', ' '.join(f'{l[:8]:>10}' for l in labels))
for i, row in enumerate(S):
    print(f'{labels[i][:12]:<12}', ' '.join(f'{v_:>10.3f}' for v_ in row))

# InfoNCE Loss
def softmax(x): return np.exp(x - x.max()) / np.exp(x - x.max()).sum()

def infonce_loss(S):
    N = S.shape[0]
    targets = np.arange(N)
    L_it = -np.mean([np.log(softmax(S[i,:])[i]) for i in range(N)])  # image->text
    L_ti = -np.mean([np.log(softmax(S[:,j])[j]) for j in range(N)])  # text->image
    return (L_it + L_ti) / 2

loss = infonce_loss(S)
print(f'\nInfoNCE Loss: {loss:.4f}')
print(f'(Lower is better; random baseline ~ log(N) = {np.log(N):.3f})')

# Visualise similarity matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

ax = axes[0]; ax.set_facecolor('#13131f')
im = ax.imshow(S, cmap='RdYlGn', vmin=S.min(), vmax=S.max())
short = [l[:8] for l in labels]
ax.set_xticks(range(N)); ax.set_xticklabels(short, color='white', rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(N)); ax.set_yticklabels(short, color='white', fontsize=9)
ax.set_title('CLIP Similarity Matrix\n(diagonal = correct pairs, should be highest)', color='white')
for i in range(N):
    for j in range(N):
        c = 'black' if S[i,j] > S.mean() else 'white'
        ax.text(j, i, f'{S[i,j]:.2f}', ha='center', va='center', color=c, fontsize=9)
if i == j: ax.text(j, i, f'{S[i,j]:.2f}', ha='center', va='center', color='gold', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# Zero-shot classification demo
ax2 = axes[1]; ax2.set_facecolor('#13131f')
# Simulate: query image = cat, compare to 5 class prompts
classes = ['a photo of a cat', 'a photo of a dog', 'a photo of a tower', 'a photo of food', 'a photo of nature']
query_img_emb = v[0]   # cat image embedding
class_embs = np.random.randn(5, D) * 0.3
class_embs[0] += v[0]  # make cat class closest
class_embs /= np.linalg.norm(class_embs, axis=1, keepdims=True)
scores = class_embs @ query_img_emb
scores_softmax = np.exp(scores/tau) / np.exp(scores/tau).sum()
short_cls = [c.replace('a photo of a ','').replace('a photo of ','')[:12] for c in classes]
bars = ax2.barh(range(5), scores_softmax, color='#4ecdc4', alpha=0.85)
ax2.set_yticks(range(5)); ax2.set_yticklabels(short_cls, color='white')
ax2.set_xlabel('Probability (softmax over cosine scores)', color='#aaa')
ax2.set_title('Zero-Shot Classification\n(query: cat image vs text prompts)', color='white')
ax2.tick_params(colors='#aaa')
for sp in ax2.spines.values(): sp.set_color('#333')
bars[0].set_color('#fd79a8')   # highlight predicted class

plt.tight_layout()
plt.savefig('llm_basic/assets/05_clip.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('Zero-shot class scores:', dict(zip(short_cls, scores_softmax.round(3))))

## 6. Interview Q&A

**Q1: How does CLIP create 1 billion negatives per training step?**
> With a batch size of N=32,768, CLIP computes an N×N similarity matrix. There are N correct (positive) pairs on the diagonal. All other N(N-1) ≈ 1 billion off-diagonal entries are implicit negatives — each image paired with every non-matching text, and vice versa. This massive negative set is what makes contrastive learning so data-efficient compared to supervised methods.

---

**Q2: What is the role of temperature τ in InfoNCE loss?**
> Temperature τ scales the logits before softmax. Small τ → sharp distribution (model can be very confident, but gradients may vanish for 'almost correct' pairs). Large τ → soft distribution (more training signal from hard negatives, but learning is slower). CLIP treats τ as a learnable parameter, initialised to 0.07, and the model learns the optimal sharpness. Empirically, a temperature around 0.02–0.07 works best.

---

**Q3: Why does prompt engineering matter for CLIP zero-shot?**
> CLIP was trained on web captions like 'a photo of a cat playing in snow' — not just 'cat'. When doing zero-shot classification with just the word 'cat', the embedding doesn't match the training distribution well. Wrapping labels in templates like 'a photo of {class}' or 'a photo of a {class} in the wild' bridges this distribution gap and improves accuracy by 3–5% on ImageNet. Ensembling multiple prompt variants improves further.

---

**Q4: What does ALIGN show that was surprising?**
> ALIGN showed that training on 1.8 billion NOISY image-text pairs (no filtering) outperforms CLIP's 400M carefully filtered pairs. This suggests that at sufficient scale, the model learns to average out noise and still extract meaningful signal. This challenged the belief that data quality is always more important than quantity — at scale, quantity wins. This principle later influenced LLM training (Common Crawl = noisy but huge).

---

**Q5: What are the most important applications of CLIP-like models?**
> (1) **Zero-shot image classification** — classify any image without retraining. (2) **Image-text retrieval** — given a text query, retrieve matching images from a database. (3) **Text-to-image generation** — DALL-E 2/Stable Diffusion use CLIP text embeddings as conditioning. (4) **Image captioning** — generate text descriptions from image embeddings. (5) **Video understanding** — extend CLIP to video frames for action recognition. (6) **Medical AI** — BioViL extends CLIP to radiology reports + chest X-rays.